In [2]:
import requests

URL = ("https://apilux3.pcloud.com/getpublinkdownload"
       "?fileid=93128819642&code=XZnqkRJZ98VUd3rEuMXAFIsEpAtnNHoKYK8k"
       "&linkpassword=&forcedownload=1")
info = requests.get(URL).json()
link = "https://" + info["hosts"][0] + info["path"]
text = requests.get(link).text
lines = [l.strip() for l in text.splitlines() if l.strip()][:1000]
print(len(lines))
print(lines[:3])

998
["RD2E7; Travaux de reconstruction, de réparation et de réaménagement du quai Maréchal Joffre dit l'Estacade, Commune de Sète", 'Construction et installation de deux ombrières photovoltaïques sur le parking du futur siège de la Communauté de Communes Ventoux Sud', "Marché Public de Maîtrise d'Oeuvre - Programme n° 263 - Travaux 2019/2020 - Renouvellement de canalisations adduction distribution eau potable"]


In [3]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
line_embeddings = model.encode(lines, batch_size=32, show_progress_bar=True)
print(line_embeddings.shape)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(998, 1024)


In [4]:
import numpy as np

np.savetxt("vectors.tsv", line_embeddings, delimiter="\t")
with open("metadata.tsv", "w", encoding="utf-8") as f:
    for l in lines:
        f.write(l.replace("\t", " ").replace("\n", " ") + "\n")

In [5]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=5, random_state=0, n_init=10).fit(line_embeddings)
labels = km.labels_
print(np.bincount(labels))

[ 96 168 241 263 230]


In [6]:
np.savetxt("clusters_vectors.tsv", km.cluster_centers_, delimiter="\t")
with open("clusters_metadata.tsv", "w", encoding="utf-8") as f:
    for c in range(5):
        first = [lines[i] for i in np.where(labels == c)[0][:3]]
        f.write(f"Cluster {c+1}: " + " | ".join(first).replace("\t", " ") + "\n")

from google.colab import files
for n in ["vectors.tsv", "metadata.tsv", "clusters_vectors.tsv", "clusters_metadata.tsv"]:
    files.download(n)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!wc -l vectors.tsv metadata.tsv clusters_vectors.tsv clusters_metadata.tsv

     998 vectors.tsv
     998 metadata.tsv
       5 clusters_vectors.tsv
       5 clusters_metadata.tsv
    2006 total


In [8]:
!zip -j fichiers_projector.zip vectors.tsv metadata.tsv clusters_vectors.tsv clusters_metadata.tsv
from google.colab import files
files.download("fichiers_projector.zip")

  adding: vectors.tsv (deflated 87%)
  adding: metadata.tsv (deflated 65%)
  adding: clusters_vectors.tsv (deflated 56%)
  adding: clusters_metadata.tsv (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>